In [17]:
import json

def getflowers(petal_length: float = 4.7) -> str:
    """
    Fetches iris / flowers info mation.

    :param petal_length: the petal length to filter species by (using less than), defaults to 4.7
    :return: a count of each flower with the petal length
    :rtype: json string
    """
    from sqlalchemy import create_engine, text
    import urllib
    import configparser; config = configparser.RawConfigParser()
    config.read('keys//keys.ini')
    
    
    s = text("""SELECT Species, COUNT(Species) count_of_species FROM iris_data 
WHERE [petal.length] < """ + str(petal_length) + """
GROUP BY Species""")

    Server = config['keys']['AzureSQLServer']
    Username = config['keys']['AzureSQLUser']
    Password = config['keys']['AzureSQLPassword']
    Database = config['keys']['AzureSQLDatabase']
    
    odbc_conn = 'Driver={ODBC Driver 17 for SQL Server};Server=tcp:' + \
        Server + f';Database={Database};Uid={Username};Pwd={Password};Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;'
    params = urllib.parse.quote_plus(odbc_conn)
    conn_str = 'mssql+pyodbc:///?odbc_connect={}'.format(params)
    engine = create_engine(conn_str)
    connection = engine.connect()
    result = connection.execute(s).fetchall()
    
    return json.dumps(str(result))


getflowers(petal_length = 3.5)


'"[(\'setosa\', 50), (\'versicolor\', 3)]"'

In [16]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from azure.ai.agents.models import FunctionTool, ToolSet
import json

# --- Build the tool definitions and toolset once ---
user_functions = {getflowers}
functions = FunctionTool(functions=user_functions)
toolset = ToolSet()
toolset.add(functions)

# --- Create client & agent ---
project = AIProjectClient(
    credential=DefaultAzureCredential(),
    endpoint="https://aifsweden.services.ai.azure.com/api/projects/AIFSwedenProject01"
)

with project:
    # Register toolset for automatic function calls during create_and_process / stream
    project.agents.enable_auto_function_calls(toolset)  # ← the missing piece
    # (Alternative: skip this line and pass toolset=toolset to create_and_process below.)

    agent = project.agents.create_agent(
        model="gpt-4.1-mini",
        name="SQLAgentWithTools",
        instructions="You are a helpful assistant",
        tools=functions.definitions,  # OK to provide definitions here
    )

    # Create thread and message
    thread = project.agents.threads.create()
    project.agents.messages.create(
        thread_id=thread.id,
        role="user",
        content="Tell me all flowers that have petal length below 4.6?"
    )

    # Create & process the run (auto function calls will execute fetch_weather)
    run = project.agents.runs.create_and_process(
        thread_id=thread.id,
        agent_id=agent.id
        # Alternative if you didn't call enable_auto_function_calls:
        # , toolset=toolset
    )
    print(f"Run finished with status: {run.status}")

    # (Optional) fetch and print the latest assistant message
    messages = list(project.agents.messages.list(thread_id=thread.id))
    for m in reversed(messages):
        if m.role == "assistant":
            print(m.content[0].text.value)
            break

    # Cleanup if you want:
    project.agents.delete_agent(agent.id)

Run finished with status: RunStatus.COMPLETED
There are three types of flowers that have petal lengths below 4.6:

1. Setosa: 50 flowers
2. Versicolor: 36 flowers
3. Virginica: 1 flower

If you need more specific information about these flowers, please let me know!
